# Workshop Notebook: Intro to Group Relative Policy Optimization (GRPO)
### Objective: Optimizing Adaptive Video Streaming Bitrates without a Critic Network

Welcome to the GRPO RL Workshop! This notebook is designed for **junior software engineers** and **recent college graduates** to build a strong, foundational intuition for how Group Relative Policy Optimization works under the hood.

Unlike traditional Actor-Critic reinforcement learning algorithms (like PPO), **GRPO does away with a separate Critic network entirely**. Instead, it generates a *group* of outputs for the exact same input, evaluates their rewards, and normalizes those rewards within the group. Better-than-average choices are reinforced; below-average choices are discouraged.

---

## The Workshop Scenario: Video Bitrate Control
Imagine you are building the streaming engine for a modern video platform (like Netflix or YouTube). Your engine must continuously select a video quality tier based on fluctuating network conditions:
* **Actions:** `360p`, `720p`, `1080p`, or `4K`.
* **The Challenge:** Network bandwidth changes constantly. If your engine greedily picks a quality higher than the current network speed can handle, the video stalls to buffer, causing **catastrophic user frustration**. If it picks a quality too low, users get annoyed by blurry pixels.
* **Why static rules (`if/else`) fail:** Fixed boundaries are fragile. A brief, 1-millisecond network spike could trick an `if` statement into jumping to 4K, instantly triggering a massive buffering stall when the speed normalizes. An RL agent learns the probabilistic risk-reward trade-off of every action per environment condition.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# Set random seed for reproducibility
torch.manual_seed(42)

# 1. Define the Action Space (Streaming Qualities)
QUALITIES = ["360p", "720p", "1080p", "4K"]
NUM_ACTIONS = len(QUALITIES)
print(f"Action Space: {QUALITIES}")

## 2. The Contextual Reward Function
This function simulates the non-linear realities of network streaming:
1. **4K** requires $\ge 25$ Mbps. If selected below that, it triggers a catastrophic buffering penalty (`-2.0`).
2. **1080p** requires $\ge 10$ Mbps. Penalty is `-1.0` if it buffers.
3. **720p** is a safe mid-tier option ($\ge 5$ Mbps), but if network speed is blazing fast (e.g., 30 Mbps), the user is slightly penalized for a blurry experience.
4. **360p** is ultra-safe, never buffers, but offers low consumer satisfaction (`0.2`).

In [ ]:
def get_streaming_reward(network_speed, choice_idx):
    """
    Returns a score balancing user satisfaction vs. buffering penalties.
    """
    quality = QUALITIES[choice_idx]
    
    if quality == "4K":
        return 1.2 if network_speed >= 25 else -2.0 
        
    elif quality == "1080p":
        return 1.0 if network_speed >= 10 else -1.0
        
    elif quality == "720p":
        return 0.6 if network_speed >= 5 else -0.5
        
    else: # 360p (Ultra-safe, low resolution)
        return 0.2

## 3. The Policy Network
We use a basic two-layer neural network. It takes a single input variable (`network_speed`) and produces 4 raw scoring elements (`logits`), one for each quality option.

In [ ]:
class StreamingPolicy(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, NUM_ACTIONS)
        )
    def forward(self, x):
        return self.net(x)

policy = StreamingPolicy()
optimizer = optim.Adam(policy.parameters(), lr=0.02)
print(policy)

## 4. The Core GRPO Engine Training Loop
Pay special attention to **Step 1** (probabilistic sampling) and **Step 3** (the comparative group advantage math).

In [ ]:
# Training Conditions: Congested Cellular Network (3Mbps), Standard WiFi (12Mbps), Home Fiber (30Mbps)
network_conditions = torch.tensor([[3.0], [12.0], [30.0]])

print("Starting GRPO Engine Optimization Loop...\n")

for epoch in range(80):
    total_loss = 0
    for speed in network_conditions:
        # Step 1: Get raw output scores and treat them as a weighted multi-sided die
        logits = policy(speed)
        dist = torch.distributions.Categorical(logits=logits)
        
        # Roll the weighted die 4 times independently to get a "Brainstorming Group"
        actions = dist.sample((4,))  # group_size = 4
        
        # Step 2: Run the environment simulations and collect the group rewards
        rewards = torch.tensor([get_streaming_reward(speed.item(), act.item()) for act in actions])
        
        # Step 3: THE GRPO SECRET SAUCE
        # We do not use a separate network to guess the absolute score of a network speed.
        # We simply standardize the rewards *within* our group of 4 brainstormed ideas.
        mean_r = rewards.mean()
        std_r = rewards.std() + 1e-8
        advantages = (rewards - mean_r) / std_r
        
        # Step 4: Calculate loss and execute a backpropagation update step
        # High relative performance = positive advantage = heavily encouraged choice
        loss = -(dist.log_prob(actions) * advantages).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:02d} | Group Optimization Loss: {total_loss:.4f}")

## 5. Verification: What Rules Did GRPO Learn?
Let's see if our model successfully learned how to optimize client-side streaming profiles purely by comparing alternative strategies against each other.

In [ ]:
print("--- Production Adaptive Bitrate Rules Learned by GRPO ---")
speeds_to_test = [3.0, 12.0, 30.0]

with torch.no_grad():
    for s in speeds_to_test:
        pred = policy(torch.tensor([s]))
        best_idx = torch.argmax(pred).item()
        print(f"Current Bandwidth: {s:>4} Mbps -> Selected Strategy: {QUALITIES[best_idx]}")

--- 

## Essential Core Questions Explained for the Workshop

### Q1: The identical `speed` is passed into `policy(speed)`. Won't `dist.sample((4,))` always output 4 copies of the exact same action?
**No!** And this is a fundamental conceptual hurdle. The neural network does *not* output a final deterministic decision. Instead, it outputs raw scores (`logits`) that translate into a **probability distribution** (e.g., 10% chance for 360p, 50% chance for 720p, 35% chance for 1080p, 5% chance for 4K).

When you call `.sample((4,))`, PyTorch rolls that unevenly weighted multi-sided die **4 distinct times**. Early in training, the distribution is highly varied, so you will get highly diverse group results like `['360p', '4K', '720p', '1080p']`. As the network optimizes and grows more confident, its probabilities narrow until it stabilizes.

### Q2: Why is this group variation absolutely critical for GRPO?
If the network returned four identical elements (e.g., `['720p', '720p', '720p', '720p']`), their simulated rewards would be perfectly identical as well. When calculating the Z-score standardization step:
```python
advantages = (rewards - rewards.mean()) / std
```
The mathematical standard deviation (`std`) drops to zero. The entire mathematical update breaks down, meaning the network **learns absolutely nothing**. GRPO fundamentally depends on group peer diversity to measure what works and what fails.

### Q3: Why is GRPO considered such a major architectural optimization?
In traditional Policy Gradient algorithms (like PPO), you must train an entire **secondary neural network** called a **Critic**. The Critic's only job is to look at an environment (like 12 Mbps) and try to guess the baseline expected value of that environment. Training a secondary network consumes massive GPU resources and memory.

GRPO completely replaces the Critic network with **pure, elegant group mathematics**. By comparing candidate choices directly against the immediate average of their peers, it isolates comparative performance perfectly without needing an external baseline evaluator.